# Your first JAXCMR simulation

Run a model studying five items and recalling them in any order. You do not need a dataset or any previous Python experience.

For help getting started, follow [Local setup with VS Code](https://githubpsyche.github.io/jaxcmr/developer-setup.html) or [Google Colab setup](https://githubpsyche.github.io/jaxcmr/colab-setup.html). In VS Code, select the project's **.venv** kernel and click **Run All**. In Colab, run the **Install JAXCMR in Colab** cell first, then run the remaining code cells in order. You do not need to add or edit any code. Read the results beneath the last code cell.

## Install JAXCMR in Colab

Run this cell to install JAXCMR and its required packages in your Colab session. Wait for it to finish before continuing. If Colab asks you to restart the session, do so before running **Load the model**. Repeat this installation when you connect to a new runtime.

In VS Code, this cell skips installation and leaves your existing environment unchanged.

In [1]:
import sys

if "google.colab" in sys.modules:
    import subprocess

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "git+https://github.com/githubpsyche/jaxcmr.git",
        ]
    )

## Load the model

This cell makes JAXCMR's memory model and simulation tools available to the rest of the notebook. It does not print any results.

In [2]:
from jax import random
import jax.numpy as jnp

from jaxcmr.models.cmr import CMR
from jaxcmr.simulation import simulate_study_and_free_recall

## Choose the model settings

These settings control how the model learns, recalls items, and decides to stop. They are example values, not values fitted to participants' data. Leave them unchanged for this first run.

The last setting, `allow_repeated_recalls`, is `False`, so the model cannot recall the same item twice.

In [3]:
params = {
    "encoding_drift_rate": 0.5,
    "start_drift_rate": 0.5,
    "recall_drift_rate": 0.5,
    "shared_support": 0.05,
    "item_support": 0.25,
    "primacy_scale": 1.0,
    "primacy_decay": 0.6,
    "learning_rate": 0.5,
    "stop_probability_scale": 0.05,
    "stop_probability_growth": 0.2,
    "choice_sensitivity": 0.6,
    "learn_after_context_update": False,
    "allow_repeated_recalls": False,
}

## Simulate study and recall

The numbers 1–5 stand for five different items. The model studies them in that order and then attempts to recall them. The fixed random seed makes this example repeatable.

In [4]:
list_length = 5
present = jnp.arange(1, list_length + 1)
rng = random.PRNGKey(0)

model = CMR(list_length, params)
# This simulation function shares an interface with tasks that use trial data.
# Free recall does not use this argument, so we supply zeros.
trial = jnp.zeros_like(present)
model, recalls = simulate_study_and_free_recall(model, present, trial, rng)

print("Study order:", present)
print("Simulated recalls:", recalls)

Study order: [1 2 3 4 5]
Simulated recalls: [1 5 2 4 3]


## Read the output

**Study order** lists the items as the model studied them. **Simulated recalls** lists the items in the order the model recalled them.

With these settings and the fixed random seed, the recall sequence is `[1 5 2 4 3]`: the model recalled item 1 first, then 5, 2, 4, and 3. It recalled every item, but in a different order from study. This is one simulated recall sequence, not an average over many runs.

The output reserves five spaces for recalls. If the model stops early, unused spaces contain **0**. For example, `[1 5 0 0 0]` would mean it recalled items 1 and 5, then stopped. Zero is not a studied item.

Running this unchanged notebook again in the same environment should reproduce the sequence. Save the notebook and its output with **File → Save** in VS Code, or **File → Save a copy in Drive** in Colab.